Python code implementing VAR-LiNGAM of nominal exchange rates and relative prices. 
These scripts replicate results (for a VAR-LiNGAM with a VECM specification) included in Tables 3, 4, 5, and 6  of "Identifying the Causal Relationship between Exchange Rates and Prices" by Hironobu Nakagawa. 

The LiNGAM algorithms used in the present analysis is developed by Shimizu (2006) and Hyvärinen et al.(2010).
References:
S. Shimizu, P. O. Hoyer, A. Hyvärinen, and A. J. Kerminen.  A linear non-gaussian acyclic model for causal discovery.  Journal of Machine Learning Research, 7: 2003-2030, 2006.
A. Hyvärinen, K. Zhang, S. Shimizu, and P. O. Hoyer. Estimation of a structural vector autoregression model using non-Gaussianity. Journal of Machine Learning Research, 11: 1709-1731, 2010.

In [1]:
import numpy as np
import pandas as pd
import graphviz
from scipy import stats  # normality test（Shapiro-Wilk test）
from scipy.stats import jarque_bera
from scipy.stats import kurtosistest
import lingam
from lingam import VARLiNGAM, ICALiNGAM, DirectLiNGAM 
from lingam.utils import make_dot, print_causal_directions, print_dagc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import warnings
warnings.simplefilter('ignore')  
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)

In [2]:
#load data; choose a sheet
rawdf = pd.read_excel("ER-P-mData.xlsx",sheet_name='Data')

In [3]:
rawdf

,Unnamed: 0,seuro,sca,sden,sja,snor,sswe,sswi,suk,peuro,pca,pden,pja,pnor,pswe,pswi,puk,pus
0,1973-M01,NaN,0.999148,6.882000,301.960000,6.619,4.742000,3.71000,0.424448,NaN,22.6,13.121989,35.758756,10.980508,14.496240,37.703816,10.27815,19.536311
1,1973-M02,NaN,0.995531,6.515000,279.100000,6.235,4.581000,3.41900,0.412201,NaN,22.8,13.261090,36.053472,11.084594,14.614051,37.956934,10.34415,19.673890
2,1973-M03,NaN,0.996555,6.180000,265.260000,5.919,4.478000,3.20700,0.403714,NaN,22.8,13.338371,36.937617,11.240712,14.665657,38.265738,10.40415,19.857330
3,1973-M04,NaN,1.000610,6.219000,265.520000,5.923,4.513000,3.23800,0.402739,NaN,23.1,13.516107,37.527047,11.292751,14.770955,38.350937,10.60215,19.994910
4,1973-M05,NaN,1.000482,6.169000,264.730000,5.849,4.437000,3.17200,0.395101,NaN,23.2,13.686122,38.214715,11.344791,14.825690,38.631893,10.68016,20.132489
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619,2024-M08,0.908085,1.365162,6.776250,146.232857,10.710,10.404177,0.85732,0.774061,126.73,161.8,98.531601,109.100000,96.788641,123.174596,100.547418,133.40000,144.365081
620,2024-M09,0.900414,1.354621,6.717238,143.220556,10.610,10.227090,0.84719,0.756390,126.61,161.1,98.201235,108.900000,97.079080,123.409764,100.253885,133.50000,144.596674
621,2024-M10,0.917065,1.372974,6.841126,149.626364,10.810,10.460004,0.86059,0.765710,127.05,161.8,98.779375,109.500000,97.659957,123.689585,100.126989,134.30000,144.763145
622,2024-M11,0.940721,1.396931,7.017319,154.035556,11.050,10.897667,0.88034,0.784251,126.63,161.8,98.449009,110.000000,97.950395,124.058710,99.991864,134.60000,144.684725


In [4]:
# 🔧 Utility: Generate lag-aware labels
def generate_labels(lags):
    return [label for i in range(lags + 1)
            for label in (f'ER(t-{i})' if i > 0 else 'ER(t)',
                          f'𝜋(t-{i})' if i > 0 else '𝜋(t)')]
df = rawdf.copy()
# 🔍 Core analysis function
def analyze_country_pair(df, sn_var, pn_var, maxlags=4, n_bootstrap=500):


    # Transform raw variables
    df['ER'] = np.log(1 / df[sn_var])
    df['pforeign'] = np.log(df[pn_var])
    df['phome'] = np.log(df['pus'])
    df['𝜋'] = df['pforeign'] - df['phome']
    df['rer'] = df['ER'] + df['𝜋']
    df['drer'] = df['rer'] - df['rer'].shift(1)

      
    # Differencing (applicable for models using differenced variables) and standardization
    df['dER'] = df['ER'].diff()
    df['d𝜋'] = df['𝜋'].diff()

    # Demean series (level and index series)
    df['ER'] = (df['ER'] - df['ER'].mean()) 
    df['𝜋'] = (df['𝜋'] - df['𝜋'].mean()) 
    
    
    # Subclass the LiNGAM implementation (e.g., ICALiNGAM) and override _estimate_adjacency_matrix to 
    # estimate the instantaneous effect matrix without pruning (without using Lasso).
    # This is appropriate, considering the two-variables case here.  

    class ICALiNGAM_OLS(ICALiNGAM):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)

        def _estimate_adjacency_matrix(self, X, prior_knowledge=None):
            # copy original logic for ordering and prior_knowledge handling,
            # but call our OLS estimator instead of predict_adaptive_lasso
            if prior_knowledge is not None:
                pk = prior_knowledge.copy()
                np.fill_diagonal(pk, 0)

            B = np.zeros([X.shape[1], X.shape[1]], dtype="float64")
            for i in range(1, len(self._causal_order)):
                target = self._causal_order[i]
                predictors = self._causal_order[:i]

                if prior_knowledge is not None:
                    predictors = [p for p in predictors if pk[target, p] != 0]

                if len(predictors) == 0:
                    continue

                # OLS on all candidate predictors (no pruning)
                lr = LinearRegression()
                lr.fit(X[:, predictors], X[:, target])
                B[target, predictors] = lr.coef_

            self._adjacency_matrix = B
            return self

    

    #-----------------------
    # VAR-LiNGAM with ICALiNGAM
    #-----------------------
    # Use the subclass with VARLiNGAM
    X = df[['dER','d𝜋']].dropna()
    lingamols = ICALiNGAM_OLS()
    vlingam_ica = VARLiNGAM(lags=maxlags, lingam_model=lingamols, 
                            criterion='bic', random_state=1, prune=False)
    vlingam_ica.fit(X)

    #----------------
    # Two-regime model ( mean reverting sample belongs to regime 1)
    #-----------------
    # create index: 1 for mean reversion, 0 otherwise
    df['index'] = np.where(
    df['drer'] *  (df['rer'] - df['rer'].mean())< 0,
    1,
    0
    )
    # Create the error correction term (if VECM is applicable)
    df['ECT_lag1'] =  df['ER'].shift(1) + df['𝜋'].shift(1)

    # choose vaiables in level or first difference
    df['ER'] = df['ER'].diff()
    df['𝜋'] = df['𝜋'].diff()
   
    df = df[['ER', '𝜋','ECT_lag1', 'index']].copy()

    # Create lagged RHS variables from full sample
    
    p = vlingam_ica._lags
    for lag in range(1, p + 1):
        df[f'ER_lag{lag}'] = df['ER'].shift(lag)
        df[f'𝜋_lag{lag}'] = df['𝜋'].shift(lag)

    # Drop initial rows with NaNs due to lagging
    df_lagged = df.dropna().copy()

    # Define RHS predictors including the ECT if needed
    rhs_cols = (
    [f'ER_lag{lag}' for lag in range(1, p + 1)] +
    [f'𝜋_lag{lag}' for lag in range(1, p + 1)] + ['ECT_lag1']
    )
    # Split data by regime
    lhs_1 = df_lagged[df_lagged['index'] == 1]
    lhs_0 = df_lagged[df_lagged['index'] == 0]

    X_1 = lhs_1[rhs_cols].values
    X_0 = lhs_0[rhs_cols].values

    residuals_1 = {}
    residuals_0 = {}

    
    # Loop over target variables and fit separate models
    for target in ['ER', '𝜋']:
        y_1 = lhs_1[target].values
        y_0 = lhs_0[target].values

        # Fit model within regime 1
        model_1 = LinearRegression().fit(X_1, y_1)
        residuals_1[target] = y_1 - model_1.predict(X_1)

        # Fit model within regime 0
        model_0 = LinearRegression().fit(X_0, y_0)
        residuals_0[target] = y_0 - model_0.predict(X_0)

    # Organize into DataFrames
    resid_df_1 = pd.DataFrame(residuals_1, index=lhs_1.index)
    resid_df_0 = pd.DataFrame(residuals_0, index=lhs_0.index)

    # Strip metadata before passing to LiNGAM
    # Instantiate ICALiNGAM model
    model_lingam1 = ICALiNGAM_OLS(random_state=1, max_iter=10000)
    model_lingam0 = ICALiNGAM_OLS(random_state=1, max_iter=10000)

    # Fit the model to the residuals
    model_lingam1.fit(resid_df_1[['ER', '𝜋']])
    Bo_1 = model_lingam1.adjacency_matrix_
    causal_1 =  model_lingam1.causal_order_ 
   
    # Normality test
    Xmr = resid_df_1[['ER', '𝜋']].values
    I = np.eye(2)
    Emr = Xmr @ (I - Bo_1).T # LiNGAM structural shocks 
    shapiro0mr = stats.shapiro(Emr[:, 0])[1]
    shapiro1mr = stats.shapiro(Emr[:, 1])[1]
    shapiro_resultsmr = [shapiro0mr, shapiro1mr]

    #stat_ER, p_ER = kurtosistest(Emr[:, 0])
    #stat_pi, p_pi = kurtosistest(Emr[:, 1])
    #shapiro_resultsmr = [p_ER, p_pi]

    #jb_ER = jarque_bera(Emr[:, 0])[1]
    #jb_pi = jarque_bera(Emr[:, 1])[1]
    #shapiro_resultsmr = [jb_ER, jb_pi]

    
    model_lingam0.fit(resid_df_0[['ER', '𝜋']])
    Bo_0 = model_lingam0.adjacency_matrix_
    causal_0 =  model_lingam0.causal_order_

    MR_indep_test = model_lingam1.get_error_independence_p_values(resid_df_1[['ER', '𝜋']])

    # Number of observations in regime 1
    num_obs = len(resid_df_1)

    #---------------------
    # Bootstrap analysis for inferences
    #--------------------
    lingamols = ICALiNGAM_OLS()
    bootstrap_model_MR = VARLiNGAM(lags=vlingam_ica._lags, 
                                lingam_model=lingamols, prune=False, 
                                random_state=1)
    bootstrap_result_MR = bootstrap_model_MR.bootstrap(Xmr, n_sampling=n_bootstrap)
 
    # extract B0 matrix from the bootstrap analysis
    amm = bootstrap_result_MR.adjacency_matrices_
    b00 = [mat[0,0] for mat in amm] 
    b01 = [mat[0,1] for mat in amm] 
    b10 = [mat[1,0] for mat in amm] 
    b11 = [mat[1,1] for mat in amm] 
    B0_bootstrap = [np.median(b00), np.median(b01), np.median(b10),np.median(b11)]

    
    # Extract ER(t) ← 𝜋(t)
    from_index = 1 # index of 𝜋(t). (index:1)+(n_features:2)*(lag:0) = 1
    to_index = 0  # index of ER(t). (index:0)+(n_features:2)*(lag:0) = 0
    samples = bootstrap_result_MR.total_effects_[:, to_index, from_index]
    b0_stats = {
        'mean': np.mean(samples),
        #'median': np.median(samples),
        'std': np.std(samples),
        #'ci': (np.percentile(samples, 5), np.percentile(samples, 95))
    }
    
    # Causal directions
    labels = generate_labels(vlingam_ica._lags)
    causal_df = pd.DataFrame(bootstrap_result_MR.get_total_causal_effects(min_causal_effect=0.0001))
    causal_df['from'] = causal_df['from'].apply(lambda x: labels[x])
    causal_df['to'] = causal_df['to'].apply(lambda x: labels[x])

    # Extract base variable names
    from_base = causal_df['from'].str.extract(r'(\w+)')[0]
    to_base = causal_df['to'].str.extract(r'(\w+)')[0]

    # Split into autocorrelation and non-autocorrelation
    causal_df_auto = causal_df[from_base.eq(to_base)]
    causal_df_no_auto = causal_df[~from_base.eq(to_base)]

    # Separate filtered results
    df_er_auto = causal_df_auto[causal_df_auto['to'] == 'ER(t)']
    df_π_auto = causal_df_auto[causal_df_auto['to'] == '𝜋(t)']

    df_er = causal_df_no_auto[causal_df_no_auto['to'] == 'ER(t)']
    df_π = causal_df_no_auto[causal_df_no_auto['to'] == '𝜋(t)']

        # probabolity of bootstrapping
    prob =  bootstrap_result_MR.get_probabilities(min_causal_effect=0.0001)

    # 🖨️ Display results
    print(f"\n🌍 Analyzing pair: {sn_var}, {pn_var}")
    print("📊 Top causal directions toward ER(t):")
    if df_er.empty:
        print("Empty DataFrame\nColumns: [from, to, effect, probability]\nIndex: []")
    else:
        print(df_er.head(10).to_string(index=False))

    if df_er.empty:
        print("Empty DataFrame\nColumns: [from, to, effect, probability]\nIndex: []")
    else:
        print(df_er_auto.head(10).to_string(index=False))
    

    print("📊 Top causal directions toward 𝜋(t):")
    if df_𝜋.empty:
        print("Empty DataFrame\nColumns: [from, to, effect, probability]\nIndex: []")
    else:
        print(df_𝜋.head(10).to_string(index=False))
    if df_𝜋.empty:
        print("Empty DataFrame\nColumns: [from, to, effect, probability]\nIndex: []")
    else:
        print(df_𝜋_auto.head(10).to_string(index=False))

    return {
        'pair': f'{sn_var}-{pn_var}',
        'shapiro_test_p_valuemr': shapiro_resultsmr,
        'MR_indep_test': MR_indep_test,
        'B0_bootstrap': B0_bootstrap,
        'B0 prob': prob[0],
        'causal_df': causal_df,
        'causal_to_ER': df_er,
        'causal_to_pai': df_𝜋,
        'causal_1': causal_1,
        'causal_0': causal_0,
        'Bo_1': Bo_1,
        'obs_1': num_obs,
        'Bo_0': Bo_0,
        'b0_stats': b0_stats,



    }

In [5]:
country_pairs = [
    ('sca', 'pca'),
    ('sden', 'pden'),
    ('seuro', 'peuro'),
    ('sja', 'pja'),
    ('snor', 'pnor'),
    ('sswe', 'pswe'),
    ('sswi', 'pswi'),
    ('suk', 'puk')
]

summary_table = []

for sn_var, pn_var in country_pairs:
    result = analyze_country_pair(df, sn_var, pn_var)

    summary_table.append({
        'Pair': result['pair'],
        'Num_obs_regime1': result['obs_1'],
        'b0_stats regime1': result['b0_stats'],
        'B0_bootstrap regime1': result['B0_bootstrap'],
        'Prob B0 regime1': result['B0 prob'].flatten().tolist(),
        'Shapiro_test_p_value regime1': result['shapiro_test_p_valuemr'],
        'Indep_test_p_value regime1': result['MR_indep_test'],
        
        
        #'Causal_to_ER(t)': len(result['causal_to_ER']),
        #'Causal_to_pai(t)': len(result['causal_to_pai'])
    })


🌍 Analyzing pair: sca, pca
📊 Top causal directions toward ER(t):
  from    to    effect  probability
𝜋(t-1) ER(t)  0.090548        1.000
  𝜋(t) ER(t) -0.593346        0.932
   from    to   effect  probability
ER(t-1) ER(t) 0.242398          1.0
📊 Top causal directions toward 𝜋(t):
   from   to    effect  probability
ER(t-1) 𝜋(t)  0.007780        0.990
  ER(t) 𝜋(t) -0.060385        0.068
  from   to   effect  probability
𝜋(t-1) 𝜋(t) 0.042443          1.0

🌍 Analyzing pair: sden, pden
📊 Top causal directions toward ER(t):
  from    to    effect  probability
𝜋(t-1) ER(t)  0.459088        1.000
  𝜋(t) ER(t) -0.777200        0.952
   from    to  effect  probability
ER(t-1) ER(t) 0.12723          1.0
📊 Top causal directions toward 𝜋(t):
   from   to    effect  probability
ER(t-1) 𝜋(t) -0.004123        0.990
  ER(t) 𝜋(t) -0.071716        0.048
  from   to   effect  probability
𝜋(t-1) 𝜋(t) 0.049404        0.996

🌍 Analyzing pair: seuro, peuro
📊 Top causal directions toward ER(t):
  from    to

In [6]:
summary_df = pd.DataFrame(summary_table)
pd.set_option('display.max_columns', None)
print("\n📋 Summary Table Across Country Pairs:")
print(summary_df)


📋 Summary Table Across Country Pairs:
          Pair  Num_obs_regime1  \
0      sca-pca              285   
1    sden-pden              291   
2  seuro-peuro              147   
3      sja-pja              293   
4    snor-pnor              283   
5    sswe-pswe              278   
6    sswi-pswi              290   
7      suk-puk              292   

                                    b0_stats regime1  \
0  {'mean': -0.5547708820046569, 'std': 0.2356388...   
1  {'mean': -0.759785767013154, 'std': 0.26603559...   
2  {'mean': -1.1896372417061136, 'std': 0.2960901...   
3  {'mean': -0.2657084225452886, 'std': 0.1494392...   
4  {'mean': -0.7265287315966689, 'std': 0.3018833...   
5  {'mean': -0.5506488189037024, 'std': 0.2289655...   
6  {'mean': -1.3238771496045518, 'std': 0.2776215...   
7  {'mean': -0.28215561249338883, 'std': 0.216851...   

                   B0_bootstrap regime1           Prob B0 regime1  \
0  [0.0, -0.5678997747260763, 0.0, 0.0]  [0.0, 0.932, 0.068, 0.0]   
1 